# Exercise 10: Examples and Common Mistakes - portable notebook

This is the **portable version** of the Exercise 10 review practice
from **Machine Learning for Neuroscience**, generated from the full
interactive course notebook. It is meant for running or editing the
code in Google Colab or in a local VS Code / Jupyter setup.

The richer version -- with the four embedded activities running in the
browser -- is the published course page:
<https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_10/exercise_10.html>

In this notebook every interactive activity is replaced by a runnable
Python equivalent or a link to that page; every Python analysis cell is
kept and runnable.

The UCI Human Activity Recognition Using Smartphones dataset used in
section 4 is (c) Jorge L. Reyes-Ortiz, Davide Anguita, Alessandro Ghio,
Luca Oneto and Xavier Parra, licensed CC BY 4.0:
<https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones>
(DOI: <https://doi.org/10.24432/C54S4K>).

## Setup

This notebook imports only `numpy`, `pandas`, and `scikit-learn`. Both
are already installed on Google Colab, and in a typical
scientific-Python environment, so there is normally nothing to do here.

If one of the imports further down fails, run the next cell once (edit
the version pins if your project needs specific ones), then restart the
kernel and run the notebook from the top. The notebook also downloads a
public data file the first time it runs, so it needs internet access.

In [ ]:
# If an import below fails, uncomment and run this line once, then
# restart the kernel. Safe on Colab, VS Code and Jupyter.
# %pip install numpy pandas scikit-learn

# Exercise 10: Examples and Common Mistakes

## What this notebook covers

This is Exercise 10 of the *Machine Learning for Neuroscience* practice
series. It reviews common choices that can make a model appear more
generalizable than it really is.

In this notebook, you will:

1. keep preprocessing and feature construction away from final test participants;
2. avoid choosing models or parameters with the test set;
3. keep related observations in the same fold;
4. use metrics and training choices that match an imbalanced classification problem.

## 1. The Boundary Around the Training Data

Every exercise so far has relied on one rule: the final test participants
are only ever touched once, at the very end, to report a result. This
exercise collects the most common ways that rule gets broken in practice --
almost always by accident, and almost always in a way that makes a model
look better than it will perform on genuinely new participants.

The correct order of operations always looks like this:

<div class="ml-boundary-diagram" role="group" aria-label="Diagram: raw data leads to splitting participants, which leads to fitting preprocessing and feature selection on training data only, which leads to fitting the model, which leads to evaluating once on locked test data." style="margin:1.2rem 0;">
<div style="display:flex;flex-wrap:wrap;align-items:center;row-gap:8px;font-size:0.86rem;">
<span style="border:1px solid var(--ml-border);border-radius:8px;padding:8px 12px;background:var(--ml-surface-alt);color:var(--ml-ink);">Raw data</span>
<span style="display:inline-flex;align-items:center;gap:6px;white-space:nowrap;"><span aria-hidden="true" style="color:var(--ml-muted);">&#8594;</span><span style="border:1px solid var(--ml-border);border-radius:8px;padding:8px 12px;background:var(--ml-surface-alt);color:var(--ml-ink);">Split participants</span></span>
<span style="display:inline-flex;align-items:center;gap:6px;white-space:nowrap;"><span aria-hidden="true" style="color:var(--ml-muted);">&#8594;</span><span style="border:1px solid var(--ml-border);border-radius:8px;padding:8px 12px;background:var(--ml-surface-alt);color:var(--ml-ink);">Fit preprocessing / selection on training data</span></span>
<span style="display:inline-flex;align-items:center;gap:6px;white-space:nowrap;"><span aria-hidden="true" style="color:var(--ml-muted);">&#8594;</span><span style="border:1px solid var(--ml-border);border-radius:8px;padding:8px 12px;background:var(--ml-surface-alt);color:var(--ml-ink);">Fit model</span></span>
<span style="display:inline-flex;align-items:center;gap:6px;white-space:nowrap;"><span aria-hidden="true" style="color:var(--ml-muted);">&#8594;</span><span style="border:1px solid var(--ml-think-border-strong);border-radius:8px;padding:8px 12px;background:var(--ml-think-accent-soft);color:var(--ml-think-accent-ink);">Evaluate once on locked test data</span></span>
</div>
</div>

Excluding test rows from `model.fit()` is necessary, but it is not enough.
Test participants must also be excluded when the notebook learns *any*
quantity or makes *any* data-dependent decision, including:

- scaling values (mean, standard deviation);
- missing-value fill values;
- PCA directions;
- target-informed feature selection;
- parameter choices (like a chosen `k` or `C`);
- the final model itself.

### Which of these must not use the test participants?

Before continuing, check your own intuition. Select every operation below
that learns a quantity or makes a data-dependent decision, and therefore
must not use the final test participants.

Which operations learn quantities or make data-dependent decisions and
must therefore not use the final test participants? Select all correct
answers (as a thought exercise -- there is nothing to click here):

1. Calculating the mean and standard deviation for scaling
2. Choosing features based on their correlation with the target
3. Fitting PCA
4. Calculating median values for filling missing data
5. Selecting a model parameter from performance results
6. Fitting the final prediction model
7. Choosing in advance whether success will be summarized with F1, ROC-AUC, or another metric

<details>
<summary><strong>Answer</strong></summary>

Options 1-6 are correct. Each learns a quantity or makes a
data-dependent decision from whichever rows it sees, so none of them
may use the final test participants -- they may use training data, and
validation data where appropriate, but not the locked test set. Option
7 is not correct: a performance metric should be chosen in advance from
the scientific question and the costs of different errors, not learned
from participant data. (Choosing a metric only after inspecting which
one makes a model look best is still poor research practice, but it is
different from fitting preprocessing or a model on the test
participants.)

</details>

> **Interactive version on the course website.** It is embedded in the
> published Exercise 10 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_10/exercise_10.html>

## 2. A Leakage Laboratory

The rest of this section compares a **correct** pipeline against a
**leaky** variant for three preprocessing operations, using the same
ABIDE-II cortical-thickness table and age-regression target from
Exercises 2, 4, and 5. In every pair below, the leaky version differs from
the correct version in exactly one place: whether the operation was fit on
the full sampled cohort (train and test rows together) or on the training
rows alone.

In [ ]:
import numpy as np
import pandas as pd

PIN = "e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b"
BASE = f"https://raw.githubusercontent.com/neurohackademy/nh2020-curriculum/{PIN}/tu-machine-learning-yarkoni/data"
model_df = pd.read_csv(f"{BASE}/abide2.tsv", sep="\t")

ct_cols = [c for c in model_df.columns if c.startswith("fsCT_")]
age = model_df["age"].to_numpy(dtype="float64")
X_ct = model_df[ct_cols].to_numpy(dtype="float64")
print(f"{len(model_df)} participants, {len(ct_cols)} cortical-thickness columns")


### Scaling before vs. after the split

The **leaky** version fits the scaler on every sampled row before
splitting. The **correct** version splits first, and only ever calls
`.fit()` on the training rows.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X_ct, age, test_size=0.25, random_state=0)

# Leaky: the scaler sees every row, including the test rows, before the split.
leaky_scaler = StandardScaler().fit(X_ct)
X_leaky = leaky_scaler.transform(X_ct)
X_train_leaky, X_test_leaky = X_leaky[: len(X_train)], X_leaky[len(X_train):]

# Correct: split first, then fit the scaler on the training rows only.
correct_pipeline = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
correct_pipeline.fit(X_train, y_train)
print("Correct test R2:", round(correct_pipeline.score(X_test, y_test), 3))


### Target-informed feature selection before vs. after the split

The **leaky** version ranks features by their correlation with age using
every sampled row -- including the test rows -- before splitting. The
**correct** version uses a single `Pipeline` so the ranking is learned from
the training rows only.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

k = 20

# Leaky: the selector sees the target for every row, including the test rows.
leaky_selector = SelectKBest(f_regression, k=k).fit(X_ct, age)

# Correct: a single pipeline fit on the training rows only.
correct_selection_pipeline = Pipeline([
    ("select", SelectKBest(f_regression, k=k)),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
correct_selection_pipeline.fit(X_train, y_train)
print("Correct test R2:", round(correct_selection_pipeline.score(X_test, y_test), 3))


### PCA before vs. after the split

The same pattern applies to PCA: the **leaky** version fits the component
directions on every sampled row before splitting; the **correct** version
learns them inside a pipeline fit on the training rows only.

In [ ]:
from sklearn.decomposition import PCA

n_components = 10

# Leaky: component directions are learned from every row, including the test rows.
leaky_prep = Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=n_components, random_state=0))])
leaky_prep.fit(X_ct)

# Correct: a single pipeline fit on the training rows only.
correct_pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=n_components, random_state=0)),
    ("model", LinearRegression()),
])
correct_pca_pipeline.fit(X_train, y_train)
print("Correct test R2:", round(correct_pca_pipeline.score(X_test, y_test), 3))


### Filling missing values before vs. after the split

Cortical thickness has no missing values in this table, so to show the
code pattern we mark a small number of values in one copy of a single
column as missing (this column has no real missing values -- the pattern,
not the result, is the lesson).

In [ ]:
rng = np.random.default_rng(0)
demo_col = ct_cols[0]
demo_series = model_df[demo_col].copy()
missing_idx = rng.choice(demo_series.index, size=30, replace=False)
demo_series.loc[missing_idx] = np.nan

train_idx, test_idx = train_test_split(demo_series.index, test_size=0.25, random_state=0)

# Leaky: the fill value is calculated from every row, including the test rows.
leaky_fill_value = demo_series.median()

# Correct: the fill value is calculated from the training rows only, then
# applied unchanged to the test rows.
correct_fill_value = demo_series.loc[train_idx].median()
print("Leaky fill value:", round(leaky_fill_value, 4))
print("Correct fill value (from training rows only):", round(correct_fill_value, 4))


### What Happens When the Test Set Leaks In?

The activity below runs the same three comparisons -- scaling, feature
selection, and PCA -- at several sample sizes and five predetermined splits
of real ABIDE-II participants, so you can see the effect across many
splits rather than one.

### What Happens When the Test Set Leaks In? on the course website

The interactive activity lets you choose a preprocessing operation, a
sample size, and one of five predetermined split seeds, then compares
the correct and leaky pipelines' test MSE and R2 for that split, plus an
aggregate view across all five splits. The runnable code pairs above
cover the same three operations.

> **Interactive version on the course website.** It is embedded in the
> published Exercise 10 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_10/exercise_10.html>
> This portable notebook links to it instead of embedding it.

The activity compares the same participants and the same outer split for
every correct/leaky pair, at sample sizes of 60, 100, 250, and all 1,004
eligible participants, across five predetermined splits (never chosen
after seeing a result). Target-informed feature selection shows the
clearest inflation, especially at small sample sizes, where a 360-feature
candidate pool makes it easy for a handful of features to look predictive
by chance. Scaling shows no difference at all here -- ordinary linear
regression's predictions do not depend on how a feature was rescaled, as
long as the same rescaling is applied consistently, so this particular
leak has no effect on this particular model. That does not make the leaky
scaling code correct: the discipline of fitting every step on training
data only is what keeps an evaluation valid, independent of whether a
given case happens to show a visible effect. **Leakage makes the
evaluation invalid even when its score is similar -- or occasionally
worse -- in one particular split.**

#### Think first

- Which participants influenced this preprocessing step in the leaky version?
- Why is target-informed feature selection a more direct form of leakage than scaling?
- If the two scores are almost equal, does that make the leaky procedure valid?
- Why can small samples make this comparison more unstable from one split to the next?

## 3. Do Not Choose the Model With the Test Set

Validation and nested cross-validation were covered in Exercise 4. This
section is a short reminder, using the same K-nearest-neighbours age
predictor and the same audited results from that exercise's "Choose k
Before Revealing the Test Set" activity.

In [ ]:
# Exercise 4's own audited candidate-k results for this exact age-regression
# task -- no new model is fit here, only the three workflows' outcomes are
# tabulated: which k each workflow would have chosen, and the test R2 that
# choice locks in.
summary = pd.DataFrame([
    {"workflow": "Chosen from training performance", "k": 1, "test R2": 0.462},
    {"workflow": "Chosen from validation performance", "k": 25, "test R2": 0.651},
    {"workflow": "Chosen from test performance (wrong)", "k": 8, "test R2": 0.667},
])
summary


Choosing `k` from **training** performance always favors the most flexible
model (here, `k=1`, which memorizes the training rows) and generalizes
poorly. Choosing `k` from **test** performance looks best of all three --
but only because it was chosen to look best on that exact test set; a new
test set would not repeat the advantage. Choosing `k` from **validation**
performance, then evaluating once on a test set nothing has touched, is the
only workflow that reports an honest estimate.

- Which workflow gives an optimistic final estimate?
- Why does trying more values of `k` on the test set make the problem worse?
- What role should the test set have after a model-development plan is fixed?

## 4. Related Observations Must Stay Together

The public [UCI Human Activity Recognition Using Smartphones dataset](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones)
([DOI: 10.24432/C54S4K](https://doi.org/10.24432/C54S4K), CC BY 4.0) contains
10,299 sensor-window rows from 30 participants performing six activities
(walking, walking upstairs, walking downstairs, sitting, standing, lying),
each 2.56 seconds long with 50% overlap between consecutive windows, and
each labelled with a participant ID. The original authors already split it
by participant (no participant appears in both their train and test sets).

A row here is a sensor window, not an independent person. Randomly
splitting rows allows windows from the same participant -- and sometimes
overlapping signal segments -- to appear on both sides of the evaluation
boundary.

This is not specific to smartphone sensors. The same risk appears with:

- repeated scans of the same participant;
- longitudinal visits;
- multiple trials from one participant in a single session;
- twins or siblings who share a family;
- acquisition sites, whenever the goal is evaluating on an unseen site.

In [ ]:
import base64
import gzip
import io

# The UCI HAR compact subset (10,299 rows, 30 participants, 18
# features) embedded here so this notebook needs no repository files
# to run.
_uci_har_compact_b64 = (
    "..."
)
compact = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(_uci_har_compact_b64))))
print(f"{len(compact)} sensor windows, {compact['participant_id'].nunique()} participants, "
      f"{compact['activity_label'].nunique()} activities")
compact.head()

### Random Windows or New Participants?

Compare ordinary random-window 5-fold cross-validation against
participant-grouped 5-fold cross-validation for a K-nearest-neighbours
activity classifier, using the 18-feature subset above.

### Random Windows or New Participants? on the course website

The interactive activity lets you choose ordinary or participant-grouped
splitting and a value of `k`, see which participants land in which
fold, and inspect the confusion matrix and accuracy/macro-F1 for any
fold. The runnable cell below reproduces the same comparison for one
value of `k`.

> **Interactive version on the course website.** It is embedded in the
> published Exercise 10 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_10/exercise_10.html>
> This portable notebook links to it instead of embedding it.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

har_features = [c for c in compact.columns if c not in ("participant_id", "activity_id", "activity_label")]
X_har = compact[har_features].to_numpy(dtype="float64")
y_har = compact["activity_id"].to_numpy()
groups_har = compact["participant_id"].to_numpy()

k = 5
for name, splitter, kwargs in [
    ("Ordinary", StratifiedKFold(n_splits=5, shuffle=True, random_state=42), {}),
    ("Participant-grouped", StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42), {"groups": groups_har}),
]:
    scores = []
    for train_idx, val_idx in splitter.split(X_har, y_har, **kwargs):
        model = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))])
        model.fit(X_har[train_idx], y_har[train_idx])
        scores.append(model.score(X_har[val_idx], y_har[val_idx]))
    print(f"{name:20s} mean validation accuracy at k={k}: {np.mean(scores):.3f}")


Under ordinary random splitting, every one of the 30 participants has
windows in more than one fold -- so every fold's training rows include
windows from participants also present in that fold's validation rows.
Under participant-grouped splitting, zero participants cross folds by
construction. Across every predetermined value of `k` (1, 3, 5, 11, 25),
ordinary splitting reports 0.03-0.09 higher accuracy and macro-F1 than
grouped splitting -- consistently across folds, not because of one unusual
fold.

#### Think first

- What is the independent experimental unit here: a sensor window, or a participant?
- Why can windows from the same person be unusually similar to each other?
- How does the 50% overlap between windows add to this risk?
- What is the neuroscience equivalent of a participant ID in this activity?

## 5. Class Imbalance Changes the Question

Using the same ABIDE-II autism-classification data and logistic-regression
pipeline as Exercise 3, this section fixes one deterministic imbalanced
cohort: 400 participants sampled to be 90% control and 10% autism, with a
correctly stratified, untouched evaluation set at the same prevalence.
Both models below use identical, training-only preprocessing -- this is not
a leakage comparison, only a comparison of two valid ways to handle
imbalance.

### High Accuracy Can Still Miss the Minority Class

Compare ordinary logistic regression against logistic regression with
`class_weight="balanced"` at any classification threshold you choose.

### High Accuracy Can Still Miss the Minority Class on the course website

The interactive activity lets you choose ordinary or class-weighted
logistic regression and any classification threshold, and see the
confusion matrix, accuracy, balanced accuracy, ROC-AUC, PR-AUC, F1,
precision, and recall update together, recomputed from fixed
predicted probabilities. The runnable cell below reproduces the same
comparison at the default threshold.

> **Interactive version on the course website.** It is embedded in the
> published Exercise 10 page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_10/exercise_10.html>
> This portable notebook links to it instead of embedding it.

In [ ]:
from sklearn.linear_model import LogisticRegression

diagnosis = (model_df["group"].to_numpy() == 1).astype(int)  # 1 = autism, 0 = control
rng = np.random.default_rng(20004)  # same cohort draw as the interactive activity's 90:10 case
majority_idx = np.flatnonzero(diagnosis == 0)
minority_idx = np.flatnonzero(diagnosis == 1)
cohort_idx = np.concatenate([
    rng.choice(majority_idx, size=360, replace=False),
    rng.choice(minority_idx, size=40, replace=False),
])
X_cohort, y_cohort = X_ct[cohort_idx], diagnosis[cohort_idx]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_cohort, y_cohort, test_size=0.25, random_state=0, stratify=y_cohort)

for name, class_weight in [("Ordinary", None), ("Class-weighted", "balanced")]:
    clf = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(C=1.0, max_iter=5000, class_weight=class_weight))])
    clf.fit(Xc_train, yc_train)
    acc = clf.score(Xc_test, yc_test)
    recall = clf.predict(Xc_test)[yc_test == 1].mean()
    print(f"{name:15s} test accuracy: {acc:.2f}   autism cases correctly flagged: {recall:.0%}")


Class weighting can improve minority-class recall or balanced accuracy
while reducing raw accuracy, but it is not guaranteed to improve every
metric -- at the default threshold, both models here catch very few autism
cases, and only at a lower threshold does class weighting clearly catch
more of them, at the cost of more false alarms. Whether that trade-off is
"better" depends on the scientific objective and the relative costs of a
missed case versus a false alarm.

- Which model detects more autistic participants at the threshold you chose?
- Which model looks better if you inspect accuracy alone?
- Is 94% accuracy useful when the majority baseline is 90%?
- Which kind of error -- a missed autism case, or a false alarm -- matters more for the intended application?

## 6. Does the Split Match the Scientific Question? (Bonus)

ABIDE-II pools participants from 17 acquisition sites. Two evaluation
designs answer two different scientific questions:

<div class="ml-site-diagram" role="group" aria-label="Diagram: random participant splitting keeps participants from every site in both training and test; leave-one-site-out splitting holds out an entire site for testing." style="margin:1.2rem 0;display:flex;flex-wrap:wrap;gap:20px;">
<div style="flex:1;min-width:220px;border:1px solid var(--ml-border);border-radius:10px;padding:10px 12px;">
<div style="font-weight:600;color:var(--ml-ink);margin-bottom:6px;font-size:0.86rem;">Random participant split</div>
<div style="display:flex;gap:3px;flex-wrap:wrap;margin-bottom:4px;">
<span style="width:14px;height:14px;border-radius:3px;background:#a9c2e3;"></span><span style="width:14px;height:14px;border-radius:3px;background:#e6c368;"></span><span style="width:14px;height:14px;border-radius:3px;background:#a3cbae;"></span><span style="width:14px;height:14px;border-radius:3px;background:#e0a877;"></span><span style="width:14px;height:14px;border-radius:3px;background:#a9c2e3;"></span><span style="width:14px;height:14px;border-radius:3px;background:#e6c368;"></span>
</div>
<div style="font-size:0.8rem;color:var(--ml-muted);">Training -- familiar sites</div>
<div style="display:flex;gap:3px;flex-wrap:wrap;margin:4px 0;">
<span style="width:14px;height:14px;border-radius:3px;background:#a3cbae;"></span><span style="width:14px;height:14px;border-radius:3px;background:#e0a877;"></span>
</div>
<div style="font-size:0.8rem;color:var(--ml-muted);">Test -- the same sites, new participants</div>
</div>
<div style="flex:1;min-width:220px;border:1px solid var(--ml-think-border-strong);border-radius:10px;padding:10px 12px;">
<div style="font-weight:600;color:var(--ml-ink);margin-bottom:6px;font-size:0.86rem;">Leave-one-site-out split</div>
<div style="display:flex;gap:3px;flex-wrap:wrap;margin-bottom:4px;">
<span style="width:14px;height:14px;border-radius:3px;background:#a9c2e3;"></span><span style="width:14px;height:14px;border-radius:3px;background:#e6c368;"></span><span style="width:14px;height:14px;border-radius:3px;background:#a3cbae;"></span><span style="width:14px;height:14px;border-radius:3px;background:#a9c2e3;"></span><span style="width:14px;height:14px;border-radius:3px;background:#e6c368;"></span>
</div>
<div style="font-size:0.8rem;color:var(--ml-muted);">Training -- 16 sites</div>
<div style="display:flex;gap:3px;flex-wrap:wrap;margin:4px 0;">
<span style="width:14px;height:14px;border-radius:3px;background:#e0a877;"></span>
</div>
<div style="font-size:0.8rem;color:var(--ml-muted);">Test -- 1 held-out site, entirely unseen</div>
</div>
</div>

Neither design is universally correct. Random participant splitting
estimates generalization to new participants from already-familiar sites;
leave-one-site-out estimates generalization to a completely new scanner and
site. Which one answers your scientific question depends on what future
population the model must serve.

## 7. Find the Mistake

Each fragment below has exactly one mistake from this notebook. For each
one: what crossed the intended evaluation boundary, why might the reported
score be misleading, and where should the operation happen instead?

**Fragment 1**

```python
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)
model = LinearRegression().fit(X_train, y_train)
```

<details>
<summary><strong>Answer</strong></summary>

The scaler is fit on `X` -- every row, including what will become the test
rows -- before the split. Fit the scaler after splitting, on `X_train`
only, inside a `Pipeline`.

</details>

**Fragment 2**

```python
X_pca = PCA(n_components=10).fit_transform(StandardScaler().fit_transform(X))
scores = cross_val_score(LinearRegression(), X_pca, y, cv=5)
```

<details>
<summary><strong>Answer</strong></summary>

Both the scaler and PCA are fit on the full `X` before cross-validation
even starts, so every fold's "held-out" rows already influenced the
component directions. Put the scaler and PCA inside a `Pipeline` and pass
that pipeline to `cross_val_score` -- then each fold refits them on that
fold's training rows only.

</details>

**Fragment 3**

```python
selector = SelectKBest(f_regression, k=20).fit(X, y)
X_selected = selector.transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.25, random_state=42)
model = LinearRegression().fit(X_train, y_train)
```

<details>
<summary><strong>Answer</strong></summary>

The feature selector uses the target `y` for every row, including the test
rows, before splitting -- the most direct form of leakage in this
notebook. Select features inside a `Pipeline` fit on the training rows
only.

</details>

**Fragment 4**

```python
X_train, X_test, y_train, y_test = train_test_split(windows, activity_labels, test_size=0.25, random_state=42)
model = KNeighborsClassifier().fit(X_train, y_train)
```

<details>
<summary><strong>Answer</strong></summary>

This splits individual sensor windows at random, with no participant
grouping -- windows from the same participant can land on both sides.
Split by participant ID instead (for example `StratifiedGroupKFold` with
participant ID as `groups`).

</details>

**Fragment 5**

```python
best_k, best_score = None, -np.inf
for k in [1, 3, 5, 11, 25]:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    score = model.score(X_test, y_test)
    if score > best_score:
        best_k, best_score = k, score
```

<details>
<summary><strong>Answer</strong></summary>

Every candidate `k` is scored directly against the test set, and the best
score on that exact test set is kept -- the reported `best_score` is
optimistic by construction. Choose `k` using a validation set or
cross-validation on the training data, then evaluate the chosen `k` once
on the test set.

</details>

## Final Checklist

- split independent participants before learning anything from the data;
- fit scaling and missing-value filling on training data;
- place feature selection and PCA inside the validation pipeline;
- tune parameters without examining the final test set;
- group families, repeated scans, visits, trials, or sensor windows appropriately;
- make the split reflect the intended future population or site;
- compare classification performance with a meaningful baseline;
- choose metrics from the scientific goal and important error types;
- use the final test set only after development decisions are fixed.